In [ ]:
%cd ../..
%load_ext autoreload
%autoreload 2

# PPCI — Ants

Prediction-Powered Causal Inference on the ants grooming dataset.

## 0. Imports & config

In [ ]:
SUBJECT = "ants"
ENCODER = "dinov2"
TOKEN   = "class"

In [ ]:
import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from scipy.stats import mannwhitneyu, sem, t as t_dist

from src.ppci import (
    PPCIDataset, build_model, train, compute_metrics,
    compute_teb_all_pairs,
    plot_outcome_distribution_ants, plot_summary, plot_comparison,
)

cfg    = OmegaConf.load(f"configs/ppci/{SUBJECT}/config.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_ds_kwargs = dict(
    outcome_cols=list(cfg.outcome.columns),
    task=cfg.outcome.task,
    env_cols=list(cfg.causal.confounders),
    env_include_treatment=True,
    n_val_videos=cfg.validation.n_videos,
    seed=cfg.training.seed,
)
outcomes = [c[len("Y_"):] for c in cfg.outcome.columns]

## 1. Dataset summary

Load a PPCIDataset from disk and inspect frame counts, treatment distribution, and outcome label balance.

In [ ]:
ANNOTATED_VERSIONS = ["v1","v2","v3","v4"]

datasets = {}
for v in ANNOTATED_VERSIONS:
    datasets[v] = PPCIDataset.from_disk(SUBJECT, v, ENCODER, TOKEN, **_ds_kwargs)
    #print(datasets[v])

In [ ]:
for ds in datasets.values():
    plot_outcome_distribution_ants(ds)

In [ ]:
obs_v3 = datasets["v3"].obs_level()
obs_v4 = datasets["v4"].obs_level()

shared_treatments = set(obs_v3["T"].tolist()).intersection(set(obs_v4["T"].tolist()))
print(f"Shared treatments between v3 and v4: {sorted(shared_treatments)}")

def mean_ci(x, confidence=0.95):
    ci = t_dist.interval(confidence, df=len(x) - 1, loc=x.mean(), scale=sem(x))
    return x.mean(), ci[0], ci[1]

for treatment in sorted(shared_treatments):
    y3 = obs_v3.loc[obs_v3["T"] == treatment, "Y_avg"].values
    y4 = obs_v4.loc[obs_v4["T"] == treatment, "Y_avg"].values

    m3, lo3, hi3 = mean_ci(y3)
    m4, lo4, hi4 = mean_ci(y4)
    stat, p_value = mannwhitneyu(y3, y4, alternative="two-sided")

    print(f"Treatment: {treatment}  (n v3={len(y3)}, n v4={len(y4)})")
    print(f"  APO v3: {m3:.4f} [{lo3:.4f}, {hi3:.4f}]")
    print(f"  APO v4: {m4:.4f} [{lo4:.4f}, {hi4:.4f}]")
    print(f"  Mann-Whitney U: {stat:.1f}, p-value: {float(p_value):.4f}")
    print("-" * 50)

## 2. Single train

- Training set: `ants/v3` 

- Test set: `ants/v4`

In [ ]:
torch.manual_seed(cfg.training.seed)
ds_train, ds_test = datasets["v3"], datasets["v4"]
best_model = train(ds_train, build_model(ds_train, cfg), cfg).to(device)

**Eval**: In-distribution

In [ ]:
m_train = compute_metrics(best_model, ds_train.X_train, ds_train.Y_train, device)
m_val   = compute_metrics(best_model, ds_train.X_val,   ds_train.Y_val,   device)
print(f"train — acc={m_train['acc']:.3f}  bacc={m_train['bacc']:.3f}  recall={m_train['recall']:.3f}  precision={m_train['precision']:.3f}")
print(f"val   — acc={m_val['acc']:.3f}  bacc={m_val['bacc']:.3f}  recall={m_val['recall']:.3f}  precision={m_val['precision']:.3f}")
plot_summary(ds_train.add_predictions(best_model, device).obs_level(), outcomes, annotations=True)

**Eval:** Out-of-distribution

In [ ]:
m_test = compute_metrics(best_model, ds_test.X, ds_test.Y, device)
print(f"test  — acc={m_test['acc']:.3f}  bacc={m_test['bacc']:.3f}  recall={m_test['recall']:.3f}  precision={m_test['precision']:.3f}")
plot_summary(ds_test.add_predictions(best_model, device).obs_level(), outcomes)

**Baseline**: always predict 0 (no grooming) — lower bound for comparison

In [ ]:
def always_zero_metrics(Y, label):
    """Metrics for a classifier that always predicts 0 (no grooming).

    By construction:
      recall    = 0      (never predicts positive → TP=0)
      precision = undef  (never predicts positive → no TP or FP)
      bacc      = 0.5    (TPR=0, TNR=1 → (0+1)/2 = 0.5, regardless of class balance)
      acc       = fraction of negative frames  (high when grooming is rare)
    """
    Y = Y.float()
    cols = Y.shape[1] if Y.dim() > 1 else 1
    acc_per_col = []
    pos_rate_per_col = []
    for c in range(cols):
        y = Y[:, c] if Y.dim() > 1 else Y
        n_neg = int((y == 0).sum())
        acc_per_col.append(n_neg / len(y))
        pos_rate_per_col.append(float(y.mean()))
    acc = sum(acc_per_col) / len(acc_per_col)
    print(f"{label:10s}  acc={acc:.3f}  bacc=0.500  recall=0.000  "
          f"| pos-rate per outcome: {[f'{r:.3f}' for r in pos_rate_per_col]}")

print("Always-0 baseline")
print("-" * 65)
always_zero_metrics(ds_train.Y_val, "val  (v3)")
always_zero_metrics(ds_test.Y,      "test (v4)")
print()
print("Trained model")
print("-" * 65)
print(f"val  (v3)   acc={m_val['acc']:.3f}  bacc={m_val['bacc']:.3f}  recall={m_val['recall']:.3f}")
print(f"test (v4)   acc={m_test['acc']:.3f}  bacc={m_test['bacc']:.3f}  recall={m_test['recall']:.3f}")
print()
print(f"Δ bacc (val):  {m_val['bacc']  - 0.5:+.3f}  above chance")
print(f"Δ bacc (test): {m_test['bacc'] - 0.5:+.3f}  above chance")

## 3. Compare methods

Vary training routine: ERM, DERM, vREx, IRM

- Training set: `ants/v3`

- Test set: `ants/v4`

In [ ]:
METHODS = ["ERM", "vREx", "IRM", "DERM"]

results_val, results_test = {}, {}
for method in METHODS:
    print(f"\n── {method} ──")
    m_cfg = OmegaConf.merge(cfg, OmegaConf.create({"training": {"method": method, "seed": 0}}))
    torch.manual_seed(0)
    best = train(ds_train, build_model(ds_train, m_cfg), m_cfg).to(device)
    m_val  = compute_metrics(best, ds_train.X_val, ds_train.Y_val, device)
    m_test = compute_metrics(best, ds_test.X, ds_test.Y, device)
    _, summary_train = compute_teb_all_pairs(best, ds_train, device, method="aipw")
    _, summary_test  = compute_teb_all_pairs(best, ds_test,  device, method="aipw")
    results_val[method]  = {"acc": m_val["acc"],  "precision": m_val["precision"],
                             "recall": m_val["recall"],  "summary_df": summary_train}
    results_test[method] = {"acc": m_test["acc"], "precision": m_test["precision"],
                             "recall": m_test["recall"], "summary_df": summary_test}

In [ ]:
fig = plot_comparison(results_val)
fig.suptitle("Train (validation|full)", y=1.02)
plt.show()

fig = plot_comparison(results_test)
fig.suptitle("Test (full|full)", y=1.02)
plt.show()

## 4. Deployment

Deployment model and new evaluation.

- Training set: `ants/v3` + `ants/v4`

- Test set: `ants/v5`

In [ ]:
# For deployment: reload train datasets with n_val_videos=0 to include all data in training
_deploy_kwargs = {**_ds_kwargs, "n_val_videos": 0}
ds_train   = PPCIDataset.concat([PPCIDataset.from_disk(SUBJECT, "v3", ENCODER, TOKEN, **_deploy_kwargs), 
                                 PPCIDataset.from_disk(SUBJECT, "v4", ENCODER, TOKEN, **_deploy_kwargs)])
ds_test    = PPCIDataset.from_disk(SUBJECT, "v5", ENCODER, TOKEN, **{**_ds_kwargs, "n_val_videos": 0})
print(ds_train)
print(ds_test)

In [ ]:
torch.manual_seed(cfg.training.seed)
best_multi = train(ds_train, build_model(ds_train, cfg), cfg).to(device)

In [ ]:
plot_summary(ds_test.add_predictions(best_multi, device).obs_level(), outcomes)